[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/sentiment_methods_ensemble_oct2026.ipynb)

# Diachronic Sentiment Analysis with an Ensemble of Models
IPHS 200 · October 2026 · Draft for class review

**Beginning Mini-Project #2: Sentiment Analysis**

This notebook begins Mini-Project #2. In class, you will run three sentiment methods on your novel, compare their emotional arcs, and investigate passages where the models agree or diverge. These explorations will help you develop the project’s central argument: **one insight about the novel that you had not considered before the analysis**, drawn from the overall arc and one or two emotional cruxes. Save your prediction, model results, passage selections, and notes so you can build on them for the project.

We will score the same text with sentimentr, TextBlob Naive Bayes, and CardiffNLP RoBERTa, compare the results, and return to the passages. The question is not just whether we can draw an arc but whether the arc helps us understand a text we know.

Before running the models, draw the emotional arc you expect. Mark a few key turning points you would expect to surface.

Adapted from Jon Chun's SentimentArcs notebook. This version compares lexical rules, a classical learned classifier, and one three-class transformer.

### Why an ensemble?
An **ensemble method** uses several models together to analyze the same data. Each model brings a different way of determining sentiment. Using them together helps us identify patterns supported across models and places where their readings diverge.

Our ensemble consists of sentimentr, TextBlob Naive Bayes, and CardiffNLP RoBERTa. We preserve and overlay their individual arcs so we can investigate both agreement and divergence. Ensemble methods can combine model outputs into an average or a vote, but here we use the models together without reducing them to one curve. Their combined evidence guides our reading of the novel and our selection of passages for closer interpretation.

### What kind of sentiment analysis are we doing?
This project uses **diachronic sentiment analysis**: we follow changes in sentiment through the ordered text of a novel to investigate its emotional arc. The horizontal axis represents progression through the text, not necessarily chronological time within the story.

- **Single-text sentiment analysis** assigns a polarity to a short text, such as a tweet: positive or negative, sometimes with a neutral category or a graded score.
- **Aspect-based sentiment analysis** distinguishes sentiment toward particular aspects or targets within a text. A restaurant review might praise the food but criticize the service.
- **Diachronic sentiment analysis** asks how sentiment changes across a longer text. We score its sentences in sequence and smooth those scores to examine the larger pattern, then read around selected regions to interpret what they reveal.

Our models score individual sentences, but the project’s question concerns the novel’s unfolding pattern. We are not assigning one positive/negative label to the whole novel or separately tracking sentiment toward each character.


## 1. Where do the scores come from?

A **lexical method** starts with a dictionary of sentiment-bearing words and rules for how nearby words change their force. A **supervised classifier** learns from examples that people have labeled. A **transformer** is also a learned model: it uses pretrained, contextual representations and can then be fine-tuned on labeled sentiment examples.

| Model | Method | Score |
|---|---|---|
| **sentimentr** | A sentiment lexicon with nearby negators, amplifiers, downtoners, and adversative conjunctions. | A graded numerical polarity score. It is not restricted to −1 to +1 and is not a probability. |
| **TextBlob NaiveBayesAnalyzer** | A classifier trained on NLTK movie reviews. It uses word-presence features and learned positive/negative associations, not sentence order. | P(positive) and P(negative). We retain both and calculate their difference for the arc. |
| **CardiffNLP RoBERTa** | A pretrained transformer fine-tuned for negative, neutral, and positive tweet sentiment. | Three class probabilities. We retain all three and calculate P(positive) − P(negative) for the arc. |

 The three methods give us different ways to interpret the same passage. 



References: [sentimentr](https://github.com/trinker/sentimentr), [TextBlob analyzers](https://textblob.readthedocs.io/en/dev/advanced_usage.html#sentiment-analyzers), [CardiffNLP model card](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest).


## 2. Setup
Run once in a fresh Colab session. If you ran the older setup, use **Runtime → Disconnect and delete runtime**, reconnect, then start here. Save your notebook and download any results first: deleting the runtime removes its temporary files. The first setup installs Python dependencies, R if necessary, and the real R sentimentr package. R packages may take several minutes to install in a fresh runtime. Students do not need to switch notebook languages. A GPU speeds up the transformer but is not required.

The class analysis uses your whole uploaded novel.

The Hugging Face packages install in a notebook-only directory so they do not downgrade Colab’s Gradio or Diffusers dependencies. The setup reports success only after checking imports.


In [ ]:
# Keep Colab's installed PyTorch and scientific packages. Install the
# Transformers 4 stack in a notebook-only directory, not site-packages.
import os
import sys
import json
import subprocess
import tempfile
import importlib
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

base_packages = ["nltk", "numpy", "pandas", "matplotlib", "plotly", "scipy", "pysbd", "textblob", "torch"]
missing = []
for package in base_packages:
    try:
        version(package)
    except PackageNotFoundError:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *missing], check=True)

hf_directory = Path(tempfile.gettempdir()) / "iphs200-sentiment-hf-v4"
hf_requirements = ["transformers==4.57.1", "huggingface-hub==0.36.0", "numpy==" + version("numpy")]
setup_signature = {"requirements": hf_requirements, "python": sys.version.split()[0]}
marker = hf_directory / "iphs200-setup.json"

# Don't mix already-imported system modules with the notebook's private stack.
for module_name in ("transformers", "huggingface_hub", "tokenizers", "safetensors"):
    module = sys.modules.get(module_name)
    if module is not None and not str(getattr(module, "__file__", "")).startswith(str(hf_directory) + os.sep):
        raise RuntimeError("Restart the Colab session (Runtime → Restart session), then rerun setup. This session has already loaded a different Hugging Face installation.")

if not marker.exists() or json.loads(marker.read_text()) != setup_signature:
    hf_directory.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "--target", str(hf_directory), "--upgrade", "--ignore-installed",
        *hf_requirements,
    ], check=True)
    marker.write_text(json.dumps(setup_signature))

if str(hf_directory) not in sys.path:
    sys.path.insert(0, str(hf_directory))
importlib.invalidate_caches()
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import transformers
import huggingface_hub
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
assert transformers.__version__ == "4.57.1"
assert huggingface_hub.__version__ == "0.36.0"
print(f"Setup ready: Transformers {transformers.__version__}, Hub {huggingface_hub.__version__}, PyTorch {torch.__version__}.")
print("Colab's system Hugging Face packages have not been replaced. Continue to the next cell.")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import tempfile
import json
import math
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import pysbd
from IPython.display import display
from scipy.signal import find_peaks, savgol_filter

SEED = 200
random.seed(SEED)
np.random.seed(SEED)
for resource in ("movie_reviews", "punkt", "punkt_tab"):
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f"Download failed: {resource}. Rerun setup after checking your connection.")


## 3. sentimentr: run the R scorer and return the scores to Python
sentimentr adjusts dictionary values using the words around them. Negators can reverse polarity; amplifiers and downtoners change its strength; “but” or “however” can change the weight of a clause. Its score is graded and can exceed −1 or +1.


In [ ]:
# Install R only in a Colab runtime when it is missing. Do not modify a local machine.
R_SCRIPT = os.environ.get("SENTIMENT_RSCRIPT") or shutil.which("Rscript")
if not R_SCRIPT:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Rscript is missing. Run in Colab, or set SENTIMENT_RSCRIPT to an installed Rscript.")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "r-base-core"], check=True)
    R_SCRIPT = shutil.which("Rscript")
subprocess.run([R_SCRIPT, "-e", 'if (!requireNamespace("sentimentr", quietly=TRUE)) install.packages("sentimentr", repos="https://cloud.r-project.org", Ncpus=2)'], check=True)
R_PACKAGE_VERSION = subprocess.check_output(
    [R_SCRIPT, "-e", 'cat(as.character(packageVersion("sentimentr")))'], text=True).strip()
print("sentimentr version:", R_PACKAGE_VERSION)

SENTIMENTR_SCRIPT = r"""
args <- commandArgs(trailingOnly=TRUE)
x <- read.csv(args[1], stringsAsFactors=FALSE, fileEncoding="UTF-8", check.names=FALSE)
stopifnot(identical(x$input_id, seq_len(nrow(x))))
split_text <- sentimentr::get_sentences(x$text)
parts <- as.data.frame(sentimentr::sentiment(split_text))
stopifnot(all(is.finite(parts$sentiment)))
# Each element_id is a row in x, even if R splits it into several sub-sentences.
averages <- aggregate(parts$sentiment, list(input_id=parts$element_id), mean)
names(averages)[2] <- "sentimentr"
counts <- aggregate(parts$sentence_id, list(input_id=parts$element_id), length)
names(counts)[2] <- "sentimentr_parts"
result <- merge(averages, counts, by="input_id", sort=TRUE)
stopifnot(identical(as.integer(result$input_id), seq_len(nrow(x))))
write.csv(result, args[2], row.names=FALSE, fileEncoding="UTF-8")
"""


def score_sentimentr(texts):
    texts = list(texts)
    if not texts or any(not isinstance(t, str) or not t.strip() for t in texts):
        raise ValueError("Supply nonempty text strings.")
    with tempfile.TemporaryDirectory() as directory:
        root = Path(directory)
        input_path, output_path = root / "input.csv", root / "output.csv"
        pd.DataFrame({"input_id": np.arange(1, len(texts) + 1), "text": texts}).to_csv(input_path, index=False)
        script_path = root / "score.R"
        script_path.write_text(SENTIMENTR_SCRIPT, encoding="utf-8")
        result = subprocess.run([R_SCRIPT, str(script_path), str(input_path), str(output_path)],
                                text=True, capture_output=True)
        if result.returncode:
            raise RuntimeError("sentimentr failed: " + result.stderr[-3000:])
        scores = pd.read_csv(output_path).sort_values("input_id")
        if not np.array_equal(scores["input_id"], np.arange(1, len(texts) + 1)):
            raise ValueError("R output is missing or duplicating input IDs.")
        if not np.isfinite(scores["sentimentr"]).all():
            raise ValueError("R returned non-finite scores.")
        return scores["sentimentr"].to_numpy(), scores["sentimentr_parts"].to_numpy()


## 4. TextBlob Naive Bayes: learning from labeled examples
We use **NaiveBayesAnalyzer**, not TextBlob’s default lexical scorer. On first use, it trains its built-in classifier on NLTK’s positive and negative movie reviews.

It learns associations between the presence of words and review labels. It does not model sentence order or the full narrative context. Its simplifying assumptions allow fast classification, but “not happy” is not automatically interpreted as a composition of negation and happiness. The movie-review domain is another limitation when reading a novel.


In [ ]:
from textblob import TextBlob
from textblob.sentiments import NaiveBayesAnalyzer

naive_bayes = NaiveBayesAnalyzer()
naive_bayes.train()  # Train once; the same trained analyzer is used for all passages.


def score_naive_bayes(texts):
    results = [naive_bayes.analyze(text) for text in texts]
    result = pd.DataFrame({"textblob_p_positive": [r.p_pos for r in results],
                           "textblob_p_negative": [r.p_neg for r in results]})
    result["textblob_nb"] = result["textblob_p_positive"] - result["textblob_p_negative"]
    return result


## 5. CardiffNLP: graded scores from three classes
CardiffNLP’s RoBERTa was pretrained on tweets and fine-tuned on TweetEval sentiment labels. It predicts **negative, neutral, and positive**. It uses the words in each input sentence as context; it does not read the whole novel before judging a sentence.


This is a probability-derived polarity score, not a model trained to regress emotional intensity. The tweet training domain may not fit a novel. Inputs over the token limit are truncated and flagged.


In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)
transformer = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, token=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer = transformer.to(device).eval()
# RoBERTa's position-embedding count includes reserved positions. Use the tokenizer's limit.
TOKEN_LIMIT = min(int(tokenizer.model_max_length), int(transformer.config.max_position_embeddings) - 2)
label_to_id = {str(label).lower(): int(i) for i, label in transformer.config.id2label.items()}
if set(label_to_id) != {"negative", "neutral", "positive"}:
    raise ValueError("Expected CardiffNLP negative/neutral/positive labels.")
print(f"Transformer device: {device}; token limit: {TOKEN_LIMIT}")


def score_cardiff(texts, batch_size=16):
    texts = list(texts)
    probabilities_all, truncated = [], []
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc="CardiffNLP batches"):
            batch = texts[start:start + batch_size]
            lengths = tokenizer(batch, truncation=False, return_length=True, verbose=False)["length"]
            inputs = tokenizer(batch, padding=True, truncation=True,
                               max_length=TOKEN_LIMIT, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            probabilities = transformer(**inputs).logits.softmax(dim=-1).cpu().numpy()
            probabilities_all.extend(probabilities)
            truncated.extend(length > TOKEN_LIMIT for length in lengths)
    probabilities = np.asarray(probabilities_all)
    result = pd.DataFrame({"cardiff_p_" + label: probabilities[:, label_to_id[label]]
                           for label in ["negative", "neutral", "positive"]})
    result["cardiff"] = result["cardiff_p_positive"] - result["cardiff_p_negative"]
    result["cardiff_truncated"] = np.asarray(truncated, dtype=bool)
    return result


## 6. Prepare the text
Upload your familiar novel as a UTF-8 plain-text file. All three methods will score the whole text after any header/footer trimming you specify below.

Inspect the beginning and end. Remove publisher material, tables of contents, and Gutenberg notices before interpreting an arc. You can set exact start/end markers below; both must occur only once. Start is included; end is excluded. Leave them blank only if the file is already prepared.

We preserve the original wording, punctuation, and case. By default we join single line breaks within paragraphs so that wrapped plain-text lines are not mistaken for sentence boundaries. Blank-line paragraph breaks remain. Turn this off if line breaks are intentional, as in verse. We then segment the text into approximate sentences. Sentence splitting can be imperfect, especially in dialogue. Sentence IDs refer to this segmentation, not printed line or page numbers.


In [ ]:
Novel_Title = "My novel" #@param {type:"string"}
START_MARKER = "" #@param {type:"string"}
END_MARKER = "" #@param {type:"string"}
Join_Soft_Linebreaks = True #@param {type:"boolean"}

from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one .txt file.")
filename, contents = next(iter(uploaded.items()))
if not filename.lower().endswith(".txt"):
    raise ValueError("Please upload a plain-text .txt file.")
raw_text = contents.decode("utf-8-sig")
text_title = Novel_Title.strip() or Path(filename).stem


def prepare_sentences(raw_text, start_marker="", end_marker="", join_soft_linebreaks=True):
    start, end = 0, len(raw_text)
    for marker in (start_marker, end_marker):
        if marker and raw_text.count(marker) != 1:
            raise ValueError(f"Marker must occur exactly once: {marker!r}")
    if start_marker:
        start = raw_text.index(start_marker)
    if end_marker:
        end = raw_text.index(end_marker)
    if end <= start:
        raise ValueError("End marker must follow start marker.")
    text = raw_text[start:end].replace("\r\n", "\n").replace("\r", "\n")
    if join_soft_linebreaks:
        text = re.sub(r"\n[ \t]+\n", "\n\n", text)
        text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    sentences = [s.strip() for s in pysbd.Segmenter(language="en", clean=False).segment(text) if s.strip()]
    if len(sentences) < 10:
        raise ValueError("Use at least 10 sentences for this arc exercise.")
    return pd.DataFrame({"sentence_id": np.arange(1, len(sentences) + 1), "text": sentences})

sentences_df = prepare_sentences(raw_text, START_MARKER, END_MARKER, Join_Soft_Linebreaks)
sentences_df["position_pct"] = np.linspace(0, 100, len(sentences_df))
print(text_title, "|", len(sentences_df), "sentences in the full selected text")
display(pd.concat([sentences_df.head(3), sentences_df.tail(3)]))
print("Check the beginning and end before proceeding. The preview is not a scoring limit.")


## 7. Score the same sentences with all three models
All models receive the same original text. R output is joined by its input IDs; sub-sentence splits are flagged. We preserve the source text rather than stripping stop words, case, or punctuation beforehand. Rerunning this cell rebuilds the score table, so scores from different novels do not accumulate. Changing smoothing later does not require rerunning the models.


In [ ]:
def score_text(sentences_df):
    result = sentences_df.copy(deep=True).reset_index(drop=True)
    texts = result["text"].tolist()
    result["sentimentr"], result["sentimentr_parts"] = score_sentimentr(texts)
    result = pd.concat([result, score_naive_bayes(texts), score_cardiff(texts)], axis=1)
    if not np.isfinite(result[["sentimentr", "textblob_nb", "cardiff"]].to_numpy()).all():
        raise ValueError("A model returned missing or non-finite scores.")
    return result

scores_df = score_text(sentences_df)
print(f"Scored {len(scores_df):,} sentences with all three models.")
print(f"CardiffNLP inputs truncated: {scores_df['cardiff_truncated'].sum():,}")
print(f"Inputs split into multiple R sub-sentences: {(scores_df['sentimentr_parts'] > 1).sum():,}")
display(scores_df.head())


## 8. Overlay the three smoothed curves
We will examine the three models together as an ensemble, keeping each model’s curve visible. Each scores the whole uploaded novel; short passages are extracted afterward to help us interpret the results.

Following *The Shapes of Stories* (pp. 30, 35–36; ensemble figures 16–17), start with a **simple moving average using a 10% sentence window**. Smooth each model’s raw scores over the same centered window, then standardize each smoothed curve by subtracting its mean and dividing by its standard deviation. **Zero means that model’s smoothed average for this text, not neutral emotion.** This puts the arcs on a common scale for comparing their shapes.

Ten percent is our starting point for novels. The book notes that 5% may be useful for unusually long novels. The code rounds the window to an odd number of sentences so it can be centered, and reports the actual window used. The book also notes weaker signal near the endpoints; this notebook leaves out positions without a full centered window.

There is no averaged ensemble line on this plot. Look at where the three curves come together and where they separate.


In [ ]:
MODEL_COLUMNS = ["sentimentr", "textblob_nb", "cardiff"]
MODEL_LABELS = {"sentimentr": "sentimentr", "textblob_nb": "TextBlob Naive Bayes", "cardiff": "CardiffNLP RoBERTa"}
Window_Percent = 10.0 #@param {type:"number"}
Smoothing = "moving average" #@param ["moving average", "Savitzky-Golay"]


def build_overlay(scores, window_percent=10.0, smoothing="moving average"):
    if not 0 < window_percent <= 50:
        raise ValueError("Choose a window percentage greater than 0 and at most 50.")
    n = len(scores)
    if n < 10:
        raise ValueError("At least 10 sentences are required.")
    raw = scores[MODEL_COLUMNS].astype(float)
    if not np.isfinite(raw.to_numpy()).all():
        raise ValueError("All three model scores must be finite for every sentence.")
    means, stds = raw.mean(), raw.std(ddof=0)
    if (stds < 1e-8).any():
        raise ValueError(f"Constant model output: {list(stds[stds < 1e-8].index)}. Inspect raw scores first.")
    window = max(3, int(round(n * window_percent / 100)))
    if window % 2 == 0:
        window += 1
    window = min(window, n if n % 2 else n - 1)
    if smoothing == "moving average":
        smooth = raw.rolling(window, center=True, min_periods=window).mean()
    elif smoothing == "Savitzky-Golay":
        smooth = pd.DataFrame(savgol_filter(raw.to_numpy(), window, 1, axis=0),
                              columns=MODEL_COLUMNS, index=scores.index)
        smooth.iloc[:window // 2] = np.nan
        smooth.iloc[-(window // 2):] = np.nan
    else:
        raise ValueError("Unknown smoothing method.")
    # Follow The Shapes of Stories: smooth first, then standardize each smoothed curve.
    smoothed_means = smooth.mean()
    smoothed_stds = smooth.std(ddof=0)
    if (smoothed_stds < 1e-8).any():
        raise ValueError("A smoothed curve is constant. Inspect the scores and window before comparing.")
    smooth = (smooth - smoothed_means) / smoothed_stds
    overlay = scores[["sentence_id", "position_pct"]].copy()
    for column in MODEL_COLUMNS:
        overlay[column] = smooth[column]
    return overlay, window, pd.DataFrame({"raw_mean": means, "raw_std": stds,
        "smoothed_mean": smoothed_means, "smoothed_std": smoothed_stds})


smooth_df, window_size, scaling_table = build_overlay(scores_df, Window_Percent, Smoothing)
print(f"Centered window: {window_size} sentences ({100 * window_size / len(scores_df):.1f}% of selected text)")
display(scaling_table)
if (scaling_table["raw_std"] < 0.05).any():
    print("One model varies little on its raw scale. Inspect its scores before interpreting its standardized curve.")


In [ ]:
import html
import textwrap
import plotly.graph_objects as go


def plot_overlay(smooth, scores, title):
    if not np.array_equal(smooth["sentence_id"], scores["sentence_id"]):
        raise ValueError("Plot scores and text must have identical sentence IDs.")
    previews = ["<br>".join(textwrap.wrap(html.escape(t[:240]), width=65)) for t in scores["text"]]
    fig = go.Figure()
    for column in MODEL_COLUMNS:
        fig.add_trace(go.Scatter(
            x=smooth["sentence_id"], y=smooth[column], name=MODEL_LABELS[column],
            mode="lines", line=dict(width=1.8), connectgaps=False,
            customdata=[[p] for p in previews],
            hovertemplate=("Sentence %{x}<br>Smoothed score: %{y:.3f}<br>%{customdata[0]}<extra>%{fullData.name}</extra>")))
    fig.add_hline(y=0, line_color="gray", line_width=0.7)
    fig.update_layout(title=title, height=550, hovermode="x unified",
                      xaxis_title="Sentence ID in selected text",
                      yaxis_title="Smoothed, standardized score (relative to each model’s smoothed mean)",
                      legend=dict(orientation="h", y=1.1))
    fig.update_xaxes(rangeslider_visible=True, showspikes=True, spikemode="across")
    fig.show()
    return fig

comparison_figure = plot_overlay(smooth_df, scores_df, text_title)


## 9. Choose three places of agreement and three of divergence
Hover over the overlaid curves to see the sentence ID, all three smoothed scores, and a preview of the text. Drag to zoom, use the range slider to move through the novel, and double-click to reset. Then identify:

- **Three places of strongest agreement.** Where do the curves come closest together or follow a similar rise, fall, peak, or valley? Explain what kind of agreement you see. Agreement near the baseline is different from agreement about a pronounced change.
- **Three places of greatest divergence.** Where are the curves farthest apart, or where does one model rise while another falls? Which models agree with one another, if any?

Choose distinct regions across the text rather than several neighboring points in the same region. Record the sentence ID shown in the hover label at the center of each selection. The computer will not choose these places for you. Check the passages and justify your choices rather than treating a visual match as proof.

NOTE: there is insufficient data at the beginning and end. While there are sophisticated ways of dealing with lack of data (e.g. time warping), for now we will not focus on these limit cases but instead on later sections with more data and stronger signal. 

**Approximate locations:** each curve is smoothed over nearby sentences. A marked peak, valley, or possible crux therefore represents a region of text, not a judgment about the single sentence at its center. Read **around** the selected sentence ID; the emotional turning point may occur earlier or later within the passage.


In [ ]:
# Enter exactly three comma-separated sentence IDs in each box after inspecting the plot.
Agreement_IDs = "" #@param {type:"string"}
Divergence_IDs = "" #@param {type:"string"}


def parse_three_ids(value):
    try:
        ids = [int(part.strip()) for part in value.split(",")]
    except ValueError:
        raise ValueError("Enter three integer sentence IDs separated by commas.")
    if len(ids) != 3 or len(set(ids)) != 3:
        raise ValueError("Choose three distinct sentence IDs for each category.")
    return ids


def record_selections(smooth, agreement_ids, divergence_ids):
    if len(set(agreement_ids + divergence_ids)) != 6:
        raise ValueError("Choose six different sentence IDs across the two categories.")
    rows = []
    for category, ids in [("agreement", agreement_ids), ("divergence", divergence_ids)]:
        for sentence_id in ids:
            row = smooth.loc[smooth["sentence_id"] == sentence_id]
            if len(row) != 1 or row[MODEL_COLUMNS].isna().any().any():
                raise ValueError(f"Sentence {sentence_id} has no full-window value. Choose a plotted location.")
            entry = row.iloc[0].to_dict()
            rows.append({"student_selection": category, **entry})
    return pd.DataFrame(rows)


if Agreement_IDs.strip() and Divergence_IDs.strip():
    selections_df = record_selections(smooth_df, parse_three_ids(Agreement_IDs), parse_three_ids(Divergence_IDs))
    display(selections_df)
else:
    selections_df = pd.DataFrame(columns=["student_selection", "sentence_id", "position_pct", *MODEL_COLUMNS])
    print("Inspect the plot, enter three agreement IDs and three divergence IDs, then rerun this cell.")


## 10. Show all six passages
After entering your six IDs above, run the next cell once. Each selection opens a passage card with the center sentence highlighted and five sentences of context on either side. Adjust `Context_Sentences` if you need more.

Each card also has expandable tables for the raw scores and CardiffNLP’s class probabilities. A separate expandable section contains the full smoothing window. That matters because the plotted value summarizes the window, not just the center sentence. Even if the selected sentence looks ordinary, a nearby passage may explain the curve.

For CardiffNLP, inspect the neutral probability as well as the polarity score. For TextBlob Naive Bayes, remember that there is no neutral class. R sub-sentence counts and transformer truncation are flagged.

**Read around the point, not just the center sentence.** With moving-average smoothing, the plotted value averages an entire window. The sentence ID is an approximate locator for that region. You must decide where the emotional crux actually occurs, if there is one, using the surrounding text.


In [ ]:
from IPython.display import HTML
Context_Sentences = 5 #@param {type:"integer"}


def extract_passage(scores, sentence_id, radius=3):
    if sentence_id not in set(scores["sentence_id"]):
        raise ValueError("Choose a sentence ID present in this text.")
    if not isinstance(radius, int) or radius < 0:
        raise ValueError("Context must be a nonnegative integer.")
    lo, hi = max(1, int(sentence_id) - radius), min(len(scores), int(sentence_id) + radius)
    return scores.loc[scores["sentence_id"].between(lo, hi),
        ["sentence_id", "text", *MODEL_COLUMNS, "textblob_p_negative", "textblob_p_positive",
         "cardiff_p_negative", "cardiff_p_neutral", "cardiff_p_positive", "cardiff_truncated", "sentimentr_parts"]].copy()


def passage_html(passage, center):
    paragraphs = []
    for row in passage.itertuples():
        text = f"[{int(row.sentence_id)}] {html.escape(row.text)}"
        if int(row.sentence_id) == center:
            text = f"<mark>{text}</mark>"
        paragraphs.append(f"<p>{text}</p>")
    return "".join(paragraphs)


def make_passage_cards(scores, smooth, selections, window, context=5):
    cards = []
    counts = {"agreement": 0, "divergence": 0}
    for selection in selections.itertuples():
        category, center = selection.student_selection, int(selection.sentence_id)
        counts[category] += 1
        local = extract_passage(scores, center, context)
        full = extract_passage(scores, center, window // 2)
        summary = smooth.loc[smooth["sentence_id"] == center, ["sentence_id", *MODEL_COLUMNS]]
        cards.append(
            f"<section style='border:1px solid #aaa;padding:16px;margin:18px 0'>"
            f"<h3>{category.title()} {counts[category]} — approximate center: sentence {center}</h3>"
            + f"<p><strong>Read around this point.</strong> The plotted values summarize the smoothing window, "
              f"sentences {int(full.sentence_id.iloc[0])}–{int(full.sentence_id.iloc[-1])}. "
              "The highlighted sentence is a locator, not an automatically identified emotional crux. "
              "Read the nearby passage and expand the full window if needed.</p>"
            + passage_html(local, center)
            + "<details><summary>Model scores and probabilities for these sentences</summary>"
            + local.to_html(index=False, escape=True, float_format=lambda v: f"{v:.3f}") + "</details>"
            + "<details><summary>Smoothed values at the selected point</summary>"
            + summary.to_html(index=False, float_format=lambda v: f"{v:.3f}") + "</details>"
            + f"<details><summary>Full smoothing window ({len(full)} sentences)</summary>"
            + passage_html(full, center) + "</details></section>")
    return "".join(cards)


if selections_df.empty:
    print("Enter your six sentence IDs in section 9 first. Then rerun this cell to show all six passages.")
else:
    display(HTML(make_passage_cards(scores_df, smooth_df, selections_df, window_size, Context_Sentences)))


## 11. Explain the agreement and divergence
For each of the six regions, record:

1. The sentence ID and what the three curves are doing.
2. A quotation from the surrounding passage.
3. Your reading of the passage and which model readings it supports, if any.
4. A possible explanation grounded in the methods: dictionary coverage, negation, word associations, review/tweet training, ambiguity, irony, speaker, or missing narrative context.

| Selection | Sentence ID | Evidence and explanation |
|---|---|---|
| Agreement 1 | | |
| Agreement 2 | | |
| Agreement 3 | | |
| Divergence 1 | | |
| Divergence 2 | | |
| Divergence 3 | | |

Does any selected region correspond to an emotional crux or turning point in your reading? Locate that moment in the surrounding text; it may fall before or after the sentence ID at the center of the smoothing window. It need not be the exact peak or valley on a curve. Explain whose emotion or experience is changing and compare with your hand-drawn prediction.

If a pattern is sensitive to smoothing, change the window and rerun sections 8–10. Do not rerun the models. Does the agreement or divergence persist, shift, or disappear?

### Which model is most useful for your text?
Do not choose a model simply because it agrees with your initial drawing or with another model. Agreement can reflect shared limitations. Use your passage comparisons to argue which model, if any, gives the most convincing account of the language for your question. Support your judgment with at least two passages, including one where the models diverge, and discuss a limitation of your preferred model. Check an ordinary passage outside your six selected regions before extending your conclusion to the novel as a whole.

You may conclude that different models are useful in different places, or that no model is clearly preferable. Keep all three curves in your analysis. Do not automatically average the two closest curves: similarity alone does not tell us that they are more accurate, and an average can hide the disagreement we need to explain.

### Begin your Mini-Project #2 interpretation
For Mini-Project #2, develop **one insight about the novel that you had not considered before this analysis**. Begin that work here with notes and passages you can develop into your project argument. Your insight may come from the overall shape of the arc, or from one or two emotional cruxes that bring something new into view.

1. State the new insight clearly. What do you now see about the novel that you had not recognized before?
2. Show the pattern that led you to it: the overall arc, and one or two crux regions. Compare with your original prediction and explain how the analysis changed or extended your reading.
3. Read around the relevant locations and quote passages that develop your interpretation. The plotted sentence ID marks the approximate center of a smoothed region; the emotional crux may occur before or after that sentence.
4. Explain why the insight matters. What does it help you understand about the novel’s characters, relationships, narrative structure, or emotional development?

Do more than describe a rise or fall or match a peak to a familiar plot event. Use the arc to make a new interpretive claim, and support it with the text. The model comparisons are preparation for this argument; they do not replace it.


## 12. Save your work
Save the raw scores, the three smoothed curves, your six selections, and the passages for those selections. The settings file records the models, versions, preprocessing, and smoothing window. Add your written interpretation to your notebook before saving your own copy.


In [ ]:
safe_title = "".join(c if c.isalnum() or c in "-_" else "_" for c in text_title)[:80] or "novel"
output_dir = Path("sentiment_results") / safe_title
output_dir.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(output_dir / f"{safe_title}_raw_scores.csv", index=False)
smooth_df.to_csv(output_dir / f"{safe_title}_smoothed_curves.csv", index=False)
selections_df.to_csv(output_dir / f"{safe_title}_student_selections.csv", index=False)

passage_rows = []
for row in selections_df.itertuples():
    context = extract_passage(scores_df, int(row.sentence_id), max(window_size // 2, Context_Sentences))
    context.insert(0, "selection_center", int(row.sentence_id))
    context.insert(0, "selection_category", row.student_selection)
    passage_rows.append(context)
if passage_rows:
    pd.concat(passage_rows, ignore_index=True).to_csv(output_dir / f"{safe_title}_selected_passages.csv", index=False)
else:
    print("No selections recorded yet. Choose your six places and rerun sections 9 and 12 to save their passages.")

run_settings = {"title": text_title, "input_mode": "upload", "sentences": len(scores_df),
    "start_marker": START_MARKER, "end_marker": END_MARKER,
    "join_soft_linebreaks": Join_Soft_Linebreaks,
    "models": MODEL_COLUMNS, "transformer_id": MODEL_ID,
    "transformer_revision": getattr(transformer.config, "_commit_hash", None),
    "sentimentr_version": R_PACKAGE_VERSION,
    "sentimentr_aggregation": "Arithmetic mean of R sub-sentences within each numbered Python sentence",
    "cardiff_score": "P(positive) minus P(negative); all three probabilities retained",
    "lexical_model": "sentimentr default dictionary and valence shifters",
    "classical_model": "TextBlob NaiveBayesAnalyzer; built-in training on NLTK movie_reviews; not PatternAnalyzer",
    "smoothing": Smoothing, "window_percent": Window_Percent, "window_sentences": window_size,
    "scaling": "10% default centered smoothing, followed by per-model z-score of the smoothed curve; no ensemble average",
    "raw_scaling": scaling_table.to_dict(), "agreement_ids": Agreement_IDs, "divergence_ids": Divergence_IDs}
from importlib.metadata import version
run_settings["package_versions"] = {package: version(package) for package in ["nltk", "pandas", "numpy", "scipy", "pysbd", "textblob", "transformers", "torch"]}
(output_dir / f"{safe_title}_settings.json").write_text(json.dumps(run_settings, indent=2), encoding="utf-8")
comparison_figure.write_html(str(output_dir / f"{safe_title}_interactive_overlay.html"), include_plotlyjs=True)
if not selections_df.empty:
    (output_dir / f"{safe_title}_passage_cards.html").write_text(
        "<!doctype html><meta charset=\"utf-8\"><title>Selected passages</title>"
        + make_passage_cards(scores_df, smooth_df, selections_df, window_size, Context_Sentences), encoding="utf-8")
archive_path = shutil.make_archive(f"{safe_title}_sentiment_results", "zip", output_dir)
print("Saved:", archive_path)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: open the zip file from the working directory.")
else:
    files.download(archive_path)
